# Introduction / Objective

The purpose of this project is to identify the first cardiac intensive care unit (CICU/CCU) stay for each hospital encounter using deidentified admission, discharge, and transfer (ADT) data.

For each `MRN_hidden` and `ENC_NO_hidden` combination, the following is identifed:

* `FIRST_ICU_ADMISSION` — the date and time of the first ICU admission.
* `FIRST_ICU_DISCHARGE` — the date and time of the end of the first ICU stay.
* `FIRST_ICU_LOS` — the length of the first ICU stay in hours.

The assignment establishes the follwing condition:
- _for ICU re-entry within one hour by treating the ICU stay as continuous. If a patient remains in the ICU without a subsequent ICU exit, October 3, 2022 is used as the discharge date, as specified in the assignment._

The final output is a dataset containing one row per encounter and is exported to an Excel file.


# Assignment Requirements

The final output must produce one row per encounter containing the following fields:

| Field                 | Description                                         |
| --------------------- | --------------------------------------------------- |
| `MRN_hidden`          | Deidentified patient identifier                     |
| `ENC_NO_hidden`       | Deidentified encounter identifier                   |
| `FIRST_ICU_ADMISSION` | First ICU admission date and time for the encounter |
| `FIRST_ICU_DISCHARGE` | First ICU discharge date and time for the encounter |
| `FIRST_ICU_LOS`       | Length of the first ICU stay in hours               |

The assignment rules:

1. Data must be grouped by `MRN_hidden` and `ENC_NO_hidden`.
2. Events must be evaluated chronologically using `ADT_DTTM_hidden`; `ENC_NO_hidden` should not be used for chronological ordering.
3. An ICU admission is identified when the patient moves to `4N-CICU`.
4. An ICU exit is identified when the patient moves from `4N-CICU` through a transfer out or discharge event.
5. If a patient leaves the ICU and returns within one hour, the ICU stay is treated as continuous.
6. If the patient returns after more than one hour, the first ICU stay ends at the original ICU exit.
7. If a patient is still in the ICU, October 3, 2022 is used as the discharge date.
8. Only the first ICU stay for each encounter is included in the final results.


## Original Variables

The following variables were provided in the original ADT dataset. Their meanings are based on the definitions provided in the assignment prompt.

| Variable          | Description                                                                                                                               |
| ----------------- | ----------------------------------------------------------------------------------------------------------------------------------------- |
| `MRN_hidden`      | Deidentified patient identifier.                                                              |
| `ENC_NO_hidden`   | Deidentified encounter identifier. The same value represents the same hospital encounter.                                                 |
| `ADT_EVNT_NM`     | Type of ADT event. Events include `ADMISSION`, `TRANSFER OUT`, and `DISCHARGE`.                                                           |
| `FROM_DEPT`       | Department the patient moved out of for an `ADMISSION` event, this represents the starting department or operating room when applicable. |
| `ADT_DTTM_hidden` | Date and time when the ADT event occurred.                                                                                                |
| `TO_DEPT`         | Department the patient moved to.                                                                                                          |


## Instructions to Identify First ICU Stay

The steps for identifying the first ICU stay derived from the prompt given are as followed below:

1. **Group by patient and encounter.**
   The data are grouped by `MRN_hidden` and `ENC_NO_hidden` because the assignment states that these two fields identify the patient and encounter.\
This ensures that ICU events from different encounters are not combined.

3. **Order events chronologically.**
   The events are arranged by `ADT_DTTM_hidden` because the assignment requires the ICU movements to be evaluated in the order they occurred.\
The assignment also specifically states that `ENC_NO_hidden` should not be used to determine chronological order.

5. **Identify ICU entry.**
   An event moving the patient to `4N-CICU` is treated as an ICU entry. This comes from the assignment's definition of an ICU admission/movement \
and the example encounters, where movement into `4N-CICU` marks the beginning of the ICU stay.

7. **Identify ICU exit.**
   An event is treated as an ICU exit when the patient moves from `4N-CICU` through a `TRANSFER OUT` or `DISCHARGE` event.\
This follows the assignment definitions for transfer and discharge events and identifies when the patient leaves the ICU.

9. **Use the first ICU entry.**
   Only the first ICU entry is used because the assignment states that multiple ICU visits can occur during the same hospitalization, but only the first ICU visit is required.

10. **Check for ICU re-entry within one hour.**
   After an ICU exit is identified, the next ICU entry is checked. The assignment states that if the patient leaves the ICU and re-enters within one hour,\
the stay should be treated as continuous. Therefore, a return within one hour does not end the first ICU stay.

12. **End the stay when re-entry is more than one hour later.**
   If the next ICU entry occurs more than one hour after the ICU exit, the first ICU stay ends at the original exit time. \
This directly follows the assignment's one-hour re-entry rule and is demonstrated by the third example encounter.

14. **Handle patients who remain in the ICU.**
   If no ICU exit occurs after the first ICU entry, October 3, 2022 is used as the discharge date. This date is explicitly specified in the assignment for patients who were still in the ICU.

15. **Calculate ICU length of stay.**
   The ICU length of stay is calculated as the difference between the first ICU discharge and first ICU admission in hours.\
The assignment requests the first ICU LOS in hours and states that rounding down is acceptable.

17. **Return one row per encounter.**
    The final results contain the patient identifier, encounter identifier, first ICU admission, first ICU discharge, and first ICU length of stay.\
This follows the assignment requirement that the final product contain one row per encounter.


## Data Preparation

The source file contains deidentified ADT records for hospital encounters. The data were grouped by patient and encounter so that ICU events can be evaluated separately for each hospitalization.

The `ADT_DTTM_hidden` field is converted from character format to a date-time format and the events are ordered chronologically within each patient and encounter.\
This allows the ICU admission and discharge events to be evaluated in the order in which they occurred.



**1. Convert `ADT_DTTM_hidden` to datetime.** \
**2. Group by `MRN_hidden` and `ENC_NO_hidden`.** \
**3. Sort events chronologically within each encounter.**

#### Convert the event date/time
This converts the data type to a date-time format to order the ecounters chronologically within each patient encounter. \
`ADT_DTTM_hidden = mdy_hm(ADT_DTTM_hidden)`

#### Group by patient and encounter
This groups by medical record number encounter number then arranged by time and date.    
`data <- data %>%` \
  `group_by(MRN_hidden, ENC_NO_hidden) %>%` \
  `arrange(ADT_DTTM_hidden, .by_group = TRUE)` 

## Identifying ICU Movements
Two logical fields are created to identify movements into and out of the cardiac intensive care unit (`4N-CICU`).
`ICU_ENTRY` identifies an event where the patient moves to the CICU. `ICU_EXIT` identifies a transfer out of or discharge from the CICU. These fields are used later to determine the first ICU stay for each encounter.

#### Identify ICU entry

Identifies an entry into the CICU. \
`data <- data %>% ` \
    `ICU_ENTRY = TO_DEPT == "4N-CICU",`
</div>    

#### Identify ICU exit
Identifies a transfer out or discharge from the CICU. \
`ICU_EXIT = FROM_DEPT == "4N-CICU" & ADT_EVNT_NM %in% c("TRANSFER OUT", "DISCHARGE")`

## Calculating the First ICU Stay

The `find_first_icu()` function calculates:
- the first ICU admission,
- first ICU discharge,
- and first ICU length of stay for each patient and encounter. 
  
The function examines ICU events chronologically. If the patient leaves the ICU and returns within one hour, the stay is treated as continuous. 

If the return occurs more than one hour after the ICU exit, the first ICU stay ends at the original exit time. 

If no ICU exit occurs after the first ICU admission, October 3, 2022 is used as the discharge date according to the assignment instructions ICU length of stay is calculated in hours and rounded down to a whole number.

#### Order Events Chronologically.
The events are arranged within the function again to ensure chronological order. \
`df <- df %>%` \
    `arrange(ADT_DTTM_hidden)`

#### Use the first ICU entry
Finds the row numbers where the patient entered the ICU. \
`entry_rows <- which(df$ICU_ENTRY)`

If the patient never entered the ICU during this encounter, return missing values for all three ICU fields.
  
  `if (length(entry_rows) == 0) {` \
    `return(tibble(` \
      `FIRST_ICU_ADMISSION = as.POSIXct(NA),` \
      `FIRST_ICU_DISCHARGE = as.POSIXct(NA),` \
      `FIRST_ICU_LOS = NA_real_))}`  
      
Uses the first ICU entry as the patient's first ICU admission. \
  `entry_index <- entry_rows[1]` \
  `admission <- df$ADT_DTTM_hidden[entry_index]` 
  
Keeps track of the current ICU entry in case the patient leaves and returns to the ICU within one hour. \
`current_entry_index <- entry_index`       



#### Identify the first ICU discharge

Continues checking ICU exits and possible ICU re-entries. Finds the next ICU exit that happens after the current ICU entry. \
  `while (TRUE) {` \
    `exit_rows <- which(` \
      `df$ICU_EXIT &` \
        `seq_len(nrow(df)) > current_entry_index)`
        
Uses the first ICU exit after the current ICU entry.
    `exit_index <- exit_rows[1]` \
    `exit_time <- df$ADT_DTTM_hidden[exit_index]`
    
Looks for another ICU entry after this ICU exit.

`next_entry_rows <- which(` \
      `df$ICU_ENTRY &` \
        `seq_len(nrow(df)) > exit_index)`
        
If there is no later ICU entry, the ICU exit we found is the end of the first ICU visit.\
    `if (length(next_entry_rows) == 0) {` \
      `discharge <- exit_time` \
      `break}`

Gets the date and time of the next ICU entry. \
    `next_entry_index <- next_entry_rows[1]` \
    `next_entry_time <- df$ADT_DTTM_hidden[next_entry_index]`
   
Calculates the number of hours between leaving the ICU and returning to the ICU.\
    `gap_hours <- as.numeric(` \
      `difftime(next_entry_time, exit_time, units = "hours"))`    


 
  
 

#### Check ICU re-entry within one hour
If the patient was outside the ICU for MORE than one hour, the first ICU visit is considered finished at the original exit.\
`gap_hours <- as.numeric(difftime(next_entry_time, exit_time, units = "hours"))`

#### Continue the stay when re-entry is within one hour
If the patient returned to the ICU within one hour, treat the ICU stay as continuous. Updates the current ICU entry and continue checking.
`if (gap_hours > 1) {` \
      `discharge <- exit_time` \
      `break}` \
      `else {`      
      `current_entry_index <- next_entry_index}` \
      `}` 

#### End the first stay when re-entry is more than one hour later
If there is no later ICU entry, the ICU exit we found is the end of the first ICU visit. This validates  the one hour re-entry reasoning. \
    `if (length(next_entry_rows) == 0) {` \
      `discharge <- exit_time` \
      `break` \
    `}`

#### Handle patients who remain in the ICU.

If there is no ICU exit after the entry, the patient is considered to still be in the ICU. The assessment says to use October 3, 2022 as the discharge date when the patient is still in the ICU.\
    `if (length(exit_rows) == 0) {`\
      `discharge <- as.POSIXct("2022-10-03 00:00:00")`\
      `break`\
    `}`

#### Calculate ICU length of stay.
Calculates the ICU length of stay in hours. \
`los <- as.numeric(`  \
    `difftime(discharge, admission, units = "hours")` \
  `)`
  
  
 


## Results and Validation

The calculated results were reviewed to confirm that the final dataset contains one row for each patient and encounter combination. Additional checks were performed to identify duplicate encounters and missing ICU results.A duplicate check was performed using `MRN_hidden` and `ENC_NO_hidden`. Missing values were also summarized for the three calculated ICU fields.


#### Calculate ICU length of stay
Calculates the number of hours between leaving the ICU and returning to the ICU. \
    `gap_hours <- as.numeric(`\
      `difftime(next_entry_time, exit_time, units = "hours"))`\
... \
Then rounds down to a whole number. \
    `FIRST_ICU_LOS = floor(los)`


#### Apply the logic to every encounter
Applied separately to every patient/encounter combination to produce the final dataset.\
`results <- data %>% group_by(MRN_hidden, ENC_NO_hidden) %>%` \
`group_modify(~ find_first_icu(.x)) %>% ungroup()`

Returns the three requested results for this encounter, wole number of hours, as required by the assessment. \
  `tibble(` \
    `FIRST_ICU_ADMISSION = admission,` \
    `FIRST_ICU_DISCHARGE = discharge,` \
    `FIRST_ICU_LOS = floor(los))`

### Validation Against Assignment Examples

The calculated results were compared with the three example encounters provided in the assignment. These examples were used to verify that the first ICU admission, first ICU discharge, and ICU length of stay were calculated according to the assignment rules, including the one-hour ICU re-entry rule.


Displays the three assignment example encounters \
    `(MRN_hidden == 203470 & ENC_NO_hidden == 29204845127) |` \
    `(MRN_hidden == 51455895 & ENC_NO_hidden == 29188820095) |` \
    `(MRN_hidden == 51455895 & ENC_NO_hidden == 29192933522)`

<div align="center">
    <img src="Screenshot 2026-09-23 233855.png" width="50%">
</div>

Along with confirmation that the final dataset contains one row for each patient and encounter combinations.
Other additional checks were performed to identify duplicate encounters and missing ICU results.
Missing values were also summarized for the three calculated ICU fields.

Checked for duplicate patient/encounter combinations. \
`results %>%` \
  `count(MRN_hidden, ENC_NO_hidden) %>%` \
  `filter(n > 1)`

Checked for missing ICU results. \
`results %>%` \
  `summarise(` \
    `missing_admission = sum(is.na(FIRST_ICU_ADMISSION)),` \
    `missing_discharge = sum(is.na(FIRST_ICU_DISCHARGE)),` \
    `missing_los = sum(is.na(FIRST_ICU_LOS)))`

Checked the number of rows in the final results. \
`nrow(results)`

Checked the final column names. \
`names(results)`

## Export Results
The validated results were exported to an Excel file. The file includes the patient and encounter identifiers along with the first ICU admission, first ICU discharge, and first ICU length of stay.


In [8]:
# Load the CSV
df <- read.csv("ICU_results.csv")
# Display the data frame
View(df)

MRN_hidden,ENC_NO_hidden,FIRST_ICU_ADMISSION,FIRST_ICU_DISCHARGE,FIRST_ICU_LOS
<int>,<dbl>,<chr>,<chr>,<int>
203470,29204845127,2022-05-19 11:54:00,2022-05-22 13:25:00,73
219133,29202551651,2022-03-18 22:12:00,2022-03-19 22:04:00,23
255606,29200807911,2022-01-28 14:40:00,2022-01-31 12:18:00,69
270484,29207707039,2022-07-30 20:13:00,2022-07-31 09:25:00,13
282052,29191878193,2021-05-26 08:45:00,2021-05-26 23:10:00,14
333221,29195513731,2021-09-08 20:22:00,2021-09-13 00:38:00,100
349711,29203421031,2022-04-11 18:11:00,2022-04-25 15:39:00,333
378901,29204494232,2022-05-08 23:20:00,2022-05-10 22:49:00,47
402609,29188888386,2021-03-06 06:39:00,2021-03-07 16:45:00,34


## Conclusion

This assignement was completed and rules were applied corrected to identify the first ICU stay for each patient encounter. The ADT data were organized by patient and encounter, ordered chronologically, and evaluated to identify ICU entry and exit events.
The task accounted for the one-hour ICU re-entry rule and the specified October 3, 2022 discharge date for encounters without a subsequent ICU exit. The final results contain one row per patient encounter with the first ICU admission, first ICU discharge, and first ICU length of stay.
The results were than validated using the three example encounters provided in the assignment. The calculated admission times, discharge times, and ICU lengths of stay matched the expected values of 73, 23, and 6 hours. A duplicate check also confirmed that there were no duplicate patient and encounter combinations in the final results.
